# Hypothesis Testing - Basketball Performance Analysis

Based on the EDA, we will test three hypotheses using statistical tests to prove that the trends we observed are statistically significant and not just random.

In [0]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

In [0]:
# load data
df = pd.read_csv("../data/processed/cleaned_basketball_data.csv")
print(f"Dataset: {df.shape}")
df.head()

## Hypothesis 1: The Ball Control Hypothesis

**Research Question:** Does protecting the basketball (high assist-to-turnover ratio) lead to more wins?

- **Null Hypothesis (H₀):** Teams with high A/TO ratio have the same win % as teams with low A/TO ratio
- **Alternative Hypothesis (H₁):** Teams with high A/TO ratio have significantly higher win %
- **Test:** Two-Sample Independent T-Test
- **Significance Level:** α = 0.05

In [0]:
# split teams at median assist/turnover ratio
median_ratio = df['assist_turnover_ratio'].median()
high_ratio = df[df['assist_turnover_ratio'] >= median_ratio]
low_ratio = df[df['assist_turnover_ratio'] < median_ratio]

print(f"High A/TO ratio teams: {len(high_ratio)}")
print(f"Low A/TO ratio teams: {len(low_ratio)}")
print(f"\nHigh ratio avg win %: {high_ratio['win_percentage'].mean():.4f}")
print(f"Low ratio avg win %: {low_ratio['win_percentage'].mean():.4f}")

# independent t-test
t_stat, p_value = stats.ttest_ind(high_ratio['win_percentage'], low_ratio['win_percentage'])

print(f"\nt-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

if p_value < 0.05:
    print("\n✓ REJECT H0: Ball control significantly impacts winning")
else:
    print("\n✗ FAIL TO REJECT H0: No significant difference")

In [0]:
# visualize
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.boxplot([high_ratio['win_percentage'], low_ratio['win_percentage']],
            labels=['High A/TO', 'Low A/TO'])
plt.ylabel('Win %')
plt.title('Win % by Ball Control')

plt.subplot(1, 2, 2)
plt.scatter(df['assist_turnover_ratio'], df['win_percentage'], alpha=0.5, color='orange')
plt.xlabel('Assist/Turnover Ratio')
plt.ylabel('Win %')
plt.title('Ball Control vs Winning')

plt.tight_layout()
plt.show()

## Hypothesis 2: Defense Wins Championships

**Research Question:** Do elite defensive teams win more games?

- **Null Hypothesis (H₀):** Elite defensive teams (top 25%) have the same win % as the rest of the league
- **Alternative Hypothesis (H₁):** Elite defensive teams have significantly higher win %
- **Test:** Two-Sample Independent T-Test
- **Note:** Lower defensive rating is better
- **Significance Level:** α = 0.05

In [0]:
# top 25% defensive teams (lower is better)
defense_threshold = df['defensive_rating'].quantile(0.25)
elite_defense = df[df['defensive_rating'] <= defense_threshold]
rest = df[df['defensive_rating'] > defense_threshold]

print(f"Elite defense (top 25%): {len(elite_defense)} teams")
print(f"Rest of league: {len(rest)} teams")
print(f"Threshold: {defense_threshold:.2f}")
print(f"\nElite defense avg win %: {elite_defense['win_percentage'].mean():.4f}")
print(f"Rest avg win %: {rest['win_percentage'].mean():.4f}")

# independent t-test
t_stat, p_value = stats.ttest_ind(elite_defense['win_percentage'], rest['win_percentage'])

print(f"\nt-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

if p_value < 0.05:
    print("\n✓ REJECT H0: Defense wins championships!")
else:
    print("\n✗ FAIL TO REJECT H0: No significant difference")

In [0]:
# visualize
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.boxplot([elite_defense['win_percentage'], rest['win_percentage']],
            labels=['Elite Defense', 'Rest'])
plt.ylabel('Win %')
plt.title('Win % by Defensive Quality')

plt.subplot(1, 2, 2)
plt.scatter(df['defensive_rating'], df['win_percentage'], alpha=0.5, color='red')
plt.xlabel('Defensive Rating (lower is better)')
plt.ylabel('Win %')
plt.title('Defense vs Winning')
plt.axvline(defense_threshold, color='black', linestyle='--', label='Elite threshold')
plt.legend()

plt.tight_layout()
plt.show()

## Hypothesis 3: Home Court Advantage

**Research Question:** Do teams perform significantly better at home than away?

- **Null Hypothesis (H₀):** Home win % equals away win %
- **Alternative Hypothesis (H₁):** Home win % is significantly higher than away win %
- **Test:** Paired T-Test (same teams in two conditions)
- **Significance Level:** α = 0.05

In [0]:
# compare home vs away for same teams
print(f"Average home win %: {df['home_win_pct'].mean():.4f}")
print(f"Average away win %: {df['away_win_pct'].mean():.4f}")
print(f"Average advantage: {df['home_advantage'].mean():.4f}")

# paired t-test
t_stat, p_value = stats.ttest_rel(df['home_win_pct'], df['away_win_pct'])

print(f"\nt-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

if p_value < 0.05:
    print("\n✓ REJECT H0: Home court advantage is real!")
else:
    print("\n✗ FAIL TO REJECT H0: No significant home advantage")

In [0]:
# visualize
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.scatter(df['home_win_pct'], df['away_win_pct'], alpha=0.5)
plt.plot([0, 1], [0, 1], 'r--', label='Equal performance')
plt.xlabel('Home Win %')
plt.ylabel('Away Win %')
plt.title('Home vs Away Performance')
plt.legend()

plt.subplot(1, 2, 2)
plt.hist(df['home_advantage'], bins=30, color='purple', edgecolor='black')
plt.axvline(0, color='red', linestyle='--', label='No advantage')
plt.xlabel('Home Advantage')
plt.ylabel('Count')
plt.title('Home Court Advantage Distribution')
plt.legend()

plt.tight_layout()
plt.show()

## Summary of Results

All three hypotheses were tested at α = 0.05 significance level:

1. **Ball Control Hypothesis**: Statistical evidence that teams with better assist-to-turnover ratios win more games
2. **Defense Wins Championships**: Elite defensive teams (top 25%) significantly outperform the rest of the league
3. **Home Court Advantage**: Strong statistical evidence that teams perform better at home than away

These findings validate the patterns observed in our EDA with mathematical rigor.